# Analyzing DHS microdata for India

India 2015-2016 - J:\DATA\DHS_PROG_DHS\IND\2015_2016

More recent is available, but probably weird due to COVID

## Documentation

DHS 7 recode manual for variable definitions: https://www.dhsprogram.com/pubs/pdf/DHSG4/Recode7_DHS_10Sep2018_DHSG4.pdf.
However, it does not state which variables are in which file.
The `.MAP` files alongside each data file list the variables in it and what they mean.

Hemoglobin variables of interest: 
- HA53: Hemoglobin level in g/dl with 1 implied dcimal
- HA54: Currently pregnant
- HA55 Result of Hemoglobin measuring.
- HA56 Hemoglobin level adjusted by altitude in g/dl with 1 implied decimal. 

Wealth index variables of interest: 
- HV270: The wealth index is a composite measure of a household's cumulative living standard.
The wealth index is calculated using easy-to-collect data on a household’s ownership of
selected assets, such as televisions and bicycles; materials used for housing construction; and
types of water access and sanitation facilities.
Generated with a statistical procedure known as principal components analysis, the wealth
index places individual households on a continuous scale of relative wealth. DHS separates
all interviewed households into five wealth quintiles to compare the influence of wealth on
various population, health and nutrition indicators. The wealth index is presented in the DHS
Final Reports and survey datasets as a background characteristic
- HV271: Wealth index factor score (5 decimals) 

Pregnancy variables of interest: 
- HML18: Pregnancy status from individual questionnaire. For complete woman’s interviews this is
taken from V213. For incomplete woman's interview with anemia testing the pregnancy
status is taken from this section.
BASE: Women with a completed individual questionnaire or when available information
from the anemia testing section.

List of datasets: https://www.dhsprogram.com/data/dataset/Nigeria_Standard-DHS_2018.cfm?flag=1

Instructions on how to calculate everything can be found at: https://www.dhsprogram.com/pubs/pdf/DHSG1/Guide_to_DHS_Statistics_DHS-7_v2.pdf

In [1]:
import pandas as pd, numpy as np

%load_ext autoreload
%autoreload 2

!date

Tue Jul 30 22:07:25 PDT 2024


## Load data, name columns

In [2]:
directory = "/snfs1/DATA/DHS_PROG_DHS/IND/2015_2016/"

### WRA

In [3]:
%%time

wra_columns = {
    "v001": "cluster_number",
    "v002": "household_number",
    "v003": "line_number",
    "v005": "weight",
    "v008": "interview_date",
    "v011": "date_of_birth",
    "v190": "wealth_quintile",
}
raw_wra_data = pd.read_stata(
    directory + "IND_DHS7_2015_2016_WN_IAIR74FL_Y2018M12D06.DTA",
    columns=wra_columns.keys(),
)

CPU times: user 43.3 s, sys: 7.53 s, total: 50.8 s
Wall time: 50.8 s


In [4]:
wra_data = raw_wra_data.copy()
wra_data

,v001,v002,v003,v005,v008,v011,v190
0,10001,1,2,191760,1387,835,middle
1,10001,1,4,191760,1387,1141,middle
2,10001,9,1,191760,1387,903,richer
3,10001,9,2,191760,1387,1129,richer
4,10001,9,3,191760,1387,1154,richer
...,...,...,...,...,...,...,...
699681,360482,61,5,2380715,1385,1023,richer
699682,360482,61,7,2380715,1385,1149,richer
699683,360482,62,1,2380715,1385,817,poorest
699684,360482,75,2,2380715,1385,1105,richer


In [5]:
wra_data = wra_data[wra_columns.keys()].rename(columns=wra_columns)

In [6]:
def recode_wealth_quintile(df):
    return df.map(
        {
            "poorest": "lowest",
            "poorer": "second",
            "middle": "middle",
            "richer": "fourth",
            "richest": "highest",
        }
    )

In [7]:
wra_data["wealth_quintile"] = recode_wealth_quintile(wra_data.wealth_quintile)

In [8]:
wra_data["weight"] = wra_data.weight / 1_000_000

### Births

In [9]:
birth_columns = {
    "v005": "weight",
    "v008": "interview_date",
    "v190": "wealth_quintile",
    "b3": "birth_date",
    "m18": "size_of_child",
    "m19": "birth_weight_kilograms",
    "s220a": "duration_of_pregnancy",
}
birth_data = pd.read_stata(
    directory + "IND_DHS7_2015_2016_BR_IABR74FL_Y2018M12D06.DTA",
    columns=birth_columns.keys(),
)
birth_data

,v005,v008,v190,b3,m18,m19,s220a
0,191760,1387,middle,1141,NaN,NaN,NaN
1,191760,1387,middle,1117,NaN,NaN,NaN
2,191760,1387,middle,1089,NaN,NaN,NaN
3,191760,1387,richer,1154,NaN,NaN,NaN
4,191760,1387,richer,1129,NaN,NaN,NaN
...,...,...,...,...,...,...,...
1315612,2380715,1385,richer,1346,very large,3250.0,9.0
1315613,2380715,1385,middle,1100,NaN,NaN,NaN
1315614,2380715,1385,middle,1059,NaN,NaN,NaN
1315615,2380715,1385,middle,1027,NaN,NaN,NaN


In [10]:
birth_data = birth_data[birth_columns.keys()].rename(columns=birth_columns)
birth_data["wealth_quintile"] = recode_wealth_quintile(birth_data.wealth_quintile)
birth_data["weight"] = birth_data.weight / 1_000_000
birth_data

,weight,interview_date,wealth_quintile,birth_date,size_of_child,birth_weight_kilograms,duration_of_pregnancy
0,0.191760,1387,middle,1141,NaN,NaN,NaN
1,0.191760,1387,middle,1117,NaN,NaN,NaN
2,0.191760,1387,middle,1089,NaN,NaN,NaN
3,0.191760,1387,fourth,1154,NaN,NaN,NaN
4,0.191760,1387,fourth,1129,NaN,NaN,NaN
...,...,...,...,...,...,...,...
1315612,2.380715,1385,fourth,1346,very large,3250.0,9.0
1315613,2.380715,1385,middle,1100,NaN,NaN,NaN
1315614,2.380715,1385,middle,1059,NaN,NaN,NaN
1315615,2.380715,1385,middle,1027,NaN,NaN,NaN


### Household members

In [11]:
%%time

hhm_columns = {
    "hv001": "cluster_number",
    "hv002": "household_number",
    "hv005": "weight",
    "hv008": "date_of_interview",
    "hvidx": "line_number",
    "hml18": "currently_pregnant",
    "hv105": "age",
    "hv104": "sex",
    "ha0": "index_to_household",
    "ha1": "age_hemoglobin",
    "hv270": "wealth_quintile",
    "ha53": "hemoglobin_raw",
    "ha56": "hemoglobin_adjusted",
    "ha57": "anemia",
}
hhm_data = pd.read_stata(
    directory + "IND_DHS7_2015_2016_HHM_IAPR74FL_Y2018M12D06.DTA",
    columns=hhm_columns.keys(),
)
hhm_data

CPU times: user 14.4 s, sys: 5.05 s, total: 19.5 s
Wall time: 23.6 s


,hv001,hv002,hv005,hv008,hvidx,hml18,hv105,hv104,ha0,ha1,hv270,ha53,ha56,ha57
0,10001,1,191072,1387,1,NaN,51,male,NaN,NaN,middle,NaN,NaN,NaN
1,10001,1,191072,1387,2,"not pregnant, don't know",46,female,2.0,46.0,middle,81.0,81.0,moderate
2,10001,1,191072,1387,3,NaN,22,male,NaN,NaN,middle,NaN,NaN,NaN
3,10001,1,191072,1387,4,"not pregnant, don't know",20,female,4.0,20.0,middle,113.0,113.0,mild
4,10001,9,191072,1387,1,"not pregnant, don't know",40,female,1.0,40.0,richer,116.0,116.0,mild
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2869038,360482,85,2270734,1385,1,NaN,60,male,NaN,NaN,middle,NaN,NaN,NaN
2869039,360482,85,2270734,1385,2,NaN,50,female,NaN,NaN,middle,NaN,NaN,NaN
2869040,360482,96,2270734,1385,1,NaN,66,male,NaN,NaN,middle,NaN,NaN,NaN
2869041,360482,96,2270734,1385,2,"not pregnant, don't know",46,female,2.0,46.0,middle,119.0,119.0,mild


In [12]:
hhm_data = hhm_data[hhm_columns.keys()].rename(columns=hhm_columns)

In [13]:
hhm_data["age"] = hhm_data.age.replace({"95+": 95, "don't know": np.nan}).astype(float)

/tmp/ipykernel_2010209/4104546349.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  hhm_data["age"] = hhm_data.age.replace({"95+": 95, "don't know": np.nan}).astype(float)
/tmp/ipykernel_2010209/4104546349.py:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  hhm_data["age"] = hhm_data.age.replace({"95+": 95, "don't know": np.nan}).astype(float)


In [14]:
# Interesting -- sometimes age is quite off.
hhm_data.loc[(hhm_data.age - hhm_data.age_hemoglobin).sort_values().index]

,cluster_number,household_number,weight,date_of_interview,line_number,currently_pregnant,age,sex,index_to_household,age_hemoglobin,wealth_quintile,hemoglobin_raw,hemoglobin_adjusted,anemia
2476814,332896,13,469645,1399,4,"not pregnant, don't know",17.0,female,4.0,48.0,richest,refused,NaN,NaN
791387,140281,68,892170,1395,4,"not pregnant, don't know",19.0,female,4.0,49.0,middle,109.0,109.0,mild
845590,140822,66,241894,1394,4,"not pregnant, don't know",20.0,female,4.0,49.0,poorer,119.0,108.0,mild
1062670,160904,16,2105279,1384,3,"not pregnant, don't know",19.0,female,3.0,48.0,richest,132.0,132.0,not anemic
2709355,340097,79,339513,1382,3,"not pregnant, don't know",16.0,female,3.0,44.0,richest,142.0,142.0,not anemic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2869037,360482,75,2270734,1385,4,NaN,2.0,male,NaN,NaN,richer,NaN,NaN,NaN
2869038,360482,85,2270734,1385,1,NaN,60.0,male,NaN,NaN,middle,NaN,NaN,NaN
2869039,360482,85,2270734,1385,2,NaN,50.0,female,NaN,NaN,middle,NaN,NaN,NaN
2869040,360482,96,2270734,1385,1,NaN,66.0,male,NaN,NaN,middle,NaN,NaN,NaN


In [15]:
(hhm_data.age - hhm_data.age_hemoglobin).describe()

count    749344.000000
mean         -0.041664
std           0.841376
min         -31.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          31.000000
dtype: float64

In [16]:
hhm_data["wealth_quintile"] = recode_wealth_quintile(hhm_data.wealth_quintile)
hhm_data["weight"] = hhm_data.weight / 1_000_000

In [17]:
for col in ["hemoglobin_raw", "hemoglobin_adjusted"]:
    hhm_data[col] = (
        hhm_data[col]
        .astype(str)
        .replace(
            {
                "not tested": np.nan,
                "not present": np.nan,
                "refused": np.nan,
                "other": np.nan,
            }
        )
        .astype(float)
    )

### Adult mortality

Adult mortality included in HH for India (recent household members who died).

In [18]:
%%time

household_columns = {
    "hv005": "weight",
    "sh70": "any_died",
    "sh71": "num_died",
    "hv270": "wealth_quintile",
}
MAX_NUM_DEATHS = 5
death_columns = {
    "sh73": "sex",
    "sh74u": "age_at_death_unit",
    "sh74n": "age_at_death",
    "sh75m": "month_of_death",
    "sh75y": "year_of_death",
    "sh76": "death_violence_or_accident",
    "sh77": "death_during_pregnancy_or_childbirth",
}
columns = list(household_columns.keys())
for death_num in range(1, MAX_NUM_DEATHS + 1):
    columns += [c + "_" + str(death_num) for c in death_columns.keys()]

adult_mortality_data = pd.read_stata(
    directory + "IND_DHS7_2015_2016_HH_IAHR74FL_Y2018M12D06.DTA", columns=columns
)
adult_mortality_data

CPU times: user 41 s, sys: 11.7 s, total: 52.8 s
Wall time: 1min 1s


,hv005,sh70,sh71,hv270,sh73_1,sh74u_1,sh74n_1,sh75m_1,sh75y_1,sh76_1,...,sh75y_4,sh76_4,sh77_4,sh73_5,sh74u_5,sh74n_5,sh75m_5,sh75y_5,sh76_5,sh77_5
0,191072,no,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,191072,no,NaN,richer,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,191072,no,NaN,richer,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,191072,no,NaN,richer,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,191072,no,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
601504,2270734,no,NaN,poorest,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
601505,2270734,no,NaN,poorer,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
601506,2270734,no,NaN,richer,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
601507,2270734,no,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
# inspired by https://stackoverflow.com/a/67393747/
adult_mortality_data_reshaped = adult_mortality_data[
    [c for c in adult_mortality_data.columns if c.split("_")[0] in death_columns.keys()]
].copy()
adult_mortality_data_reshaped.columns = adult_mortality_data_reshaped.columns.str.split(
    "_", expand=True
)
adult_mortality_data_reshaped

,sh73,sh74u,sh74n,sh75m,sh75y,sh76,sh77,sh73,sh74u,sh74n,...,sh75y,sh76,sh77,sh73,sh74u,sh74n,sh75m,sh75y,sh76,sh77
,1,1,1,1,1,1,1,2,2,2,...,4,4,4,5,5,5,5,5,5,5
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
601504,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
601505,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
601506,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
adult_mortality_data_reshaped[list(household_columns.keys())] = adult_mortality_data[
    list(household_columns.keys())
]
adult_mortality_data_reshaped

,sh73,sh74u,sh74n,sh75m,sh75y,sh76,sh77,sh73,sh74u,sh74n,...,sh74u,sh74n,sh75m,sh75y,sh76,sh77,hv005,sh70,sh71,hv270
,1,1,1,1,1,1,1,2,2,2,...,5,5,5,5,5,5,,,,
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,191072,no,NaN,middle
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,191072,no,NaN,richer
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,191072,no,NaN,richer
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,191072,no,NaN,richer
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,191072,no,NaN,middle
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
601504,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2270734,no,NaN,poorest
601505,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2270734,no,NaN,poorer
601506,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2270734,no,NaN,richer


In [21]:
# Get a row per death
adult_mortality_data_reshaped = (
    adult_mortality_data_reshaped.set_index(list(household_columns.keys()))
    .swaplevel(axis=1)
    .stack(0)
    .reset_index()
    .drop(columns=[f"level_{len(household_columns)}"])
)
adult_mortality_data_reshaped

/tmp/ipykernel_2010209/3665094636.py:5: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  .stack(0)


,hv005,sh70,sh71,hv270,sh73,sh74u,sh74n,sh75m,sh75y,sh76,sh77
0,8939,yes,1.0,middle,male,year,51.0,april,2015.0,no,NaN
1,8939,yes,1.0,middle,male,months,56.0,april,2015.0,yes,NaN
2,4495,yes,1.0,middle,male,year,45.0,august,2014.0,no,NaN
3,4495,yes,2.0,richer,male,year,35.0,may,2014.0,no,NaN
4,4495,yes,2.0,richer,male,year,40.0,august,2014.0,no,NaN
...,...,...,...,...,...,...,...,...,...,...,...
74940,2463153,yes,2.0,poorer,female,year,21.0,september,2013.0,yes,NaN
74941,2270734,yes,1.0,middle,female,year,80.0,november,2014.0,no,no
74942,2270734,yes,1.0,middle,male,year,50.0,january,2015.0,no,NaN
74943,2270734,yes,1.0,middle,male,months,45.0,january,2012.0,yes,NaN


In [22]:
adult_mortality_data = (
    adult_mortality_data_reshaped[
        list(household_columns.keys()) + list(death_columns.keys())
    ]
    .rename(columns=household_columns)
    .rename(columns=death_columns)
)
adult_mortality_data["wealth_quintile"] = recode_wealth_quintile(
    adult_mortality_data.wealth_quintile
)
adult_mortality_data["weight"] = adult_mortality_data.weight / 1_000_000
adult_mortality_data

,weight,any_died,num_died,wealth_quintile,sex,age_at_death_unit,age_at_death,month_of_death,year_of_death,death_violence_or_accident,death_during_pregnancy_or_childbirth
0,0.008939,yes,1.0,middle,male,year,51.0,april,2015.0,no,NaN
1,0.008939,yes,1.0,middle,male,months,56.0,april,2015.0,yes,NaN
2,0.004495,yes,1.0,middle,male,year,45.0,august,2014.0,no,NaN
3,0.004495,yes,2.0,fourth,male,year,35.0,may,2014.0,no,NaN
4,0.004495,yes,2.0,fourth,male,year,40.0,august,2014.0,no,NaN
...,...,...,...,...,...,...,...,...,...,...,...
74940,2.463153,yes,2.0,second,female,year,21.0,september,2013.0,yes,NaN
74941,2.270734,yes,1.0,middle,female,year,80.0,november,2014.0,no,no
74942,2.270734,yes,1.0,middle,male,year,50.0,january,2015.0,no,NaN
74943,2.270734,yes,1.0,middle,male,months,45.0,january,2012.0,yes,NaN


## WRA

### Hemoglobin among pregnancies

In [23]:
id_columns = ["cluster_number", "household_number", "line_number"]
other_overlapping_columns = (
    (set(wra_data.columns) & set(hhm_data.columns)) - set(id_columns) - {"weight"}
)
other_overlapping_columns

{'wealth_quintile'}

In [24]:
wra_hhm_joined = wra_data.merge(
    hhm_data.drop(columns=["weight"]),
    on=id_columns,
    suffixes=("_wra", "_hhm"),
    how="left",
)
wra_hhm_joined

,cluster_number,household_number,line_number,weight,interview_date,date_of_birth,wealth_quintile_wra,date_of_interview,currently_pregnant,age,sex,index_to_household,age_hemoglobin,wealth_quintile_hhm,hemoglobin_raw,hemoglobin_adjusted,anemia
0,10001,1,2,0.191760,1387,835,middle,1387,"not pregnant, don't know",46.0,female,2.0,46.0,middle,81.0,81.0,moderate
1,10001,1,4,0.191760,1387,1141,middle,1387,"not pregnant, don't know",20.0,female,4.0,20.0,middle,113.0,113.0,mild
2,10001,9,1,0.191760,1387,903,fourth,1387,"not pregnant, don't know",40.0,female,1.0,40.0,fourth,116.0,116.0,mild
3,10001,9,2,0.191760,1387,1129,fourth,1387,"not pregnant, don't know",21.0,female,2.0,21.0,fourth,137.0,137.0,not anemic
4,10001,9,3,0.191760,1387,1154,fourth,1387,"not pregnant, don't know",19.0,female,3.0,19.0,fourth,137.0,137.0,not anemic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
699681,360482,61,5,2.380715,1385,1023,fourth,1385,"not pregnant, don't know",30.0,female,5.0,30.0,fourth,125.0,125.0,not anemic
699682,360482,61,7,2.380715,1385,1149,fourth,1385,"not pregnant, don't know",20.0,female,7.0,19.0,fourth,112.0,112.0,mild
699683,360482,62,1,2.380715,1385,817,lowest,1385,"not pregnant, don't know",47.0,female,1.0,47.0,lowest,106.0,103.0,mild
699684,360482,75,2,2.380715,1385,1105,fourth,1385,"not pregnant, don't know",23.0,female,2.0,23.0,fourth,103.0,103.0,mild


In [25]:
for col in other_overlapping_columns:
    assert (wra_hhm_joined[f"{col}_wra"] == wra_hhm_joined[f"{col}_hhm"]).all()
    wra_hhm_joined[col] = wra_hhm_joined[f"{col}_wra"]
    wra_hhm_joined = wra_hhm_joined.drop(columns=[f"{col}_wra", f"{col}_hhm"])

In [26]:
col

'wealth_quintile'

In [27]:
wra_hhm_joined.currently_pregnant.value_counts(dropna=False)

currently_pregnant
not pregnant, don't know    667258
pregnant                     32428
Name: count, dtype: int64

In [28]:
pregnant_data = wra_hhm_joined[wra_hhm_joined.currently_pregnant == "pregnant"].copy()
pregnant_data

,cluster_number,household_number,line_number,weight,interview_date,date_of_birth,date_of_interview,currently_pregnant,age,sex,index_to_household,age_hemoglobin,hemoglobin_raw,hemoglobin_adjusted,anemia,wealth_quintile
103,10004,45,8,0.022873,1386,1121,1386,pregnant,22.0,female,8.0,22.0,105.0,105.0,mild,middle
157,10006,71,2,0.024436,1384,1069,1384,pregnant,26.0,female,2.0,26.0,88.0,88.0,moderate,second
160,10006,83,2,0.024436,1384,938,1384,pregnant,37.0,female,2.0,37.0,93.0,93.0,moderate,fourth
164,10007,2,2,0.001204,1384,1026,1384,pregnant,28.0,female,2.0,29.0,119.0,119.0,not anemic,highest
168,10007,23,2,0.001204,1385,1089,1385,pregnant,24.0,female,2.0,24.0,99.0,99.0,moderate,second
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
699445,360469,36,2,3.262083,1384,986,1384,pregnant,33.0,female,2.0,33.0,NaN,NaN,NaN,highest
699480,360471,51,2,2.001115,1383,1108,1383,pregnant,22.0,female,2.0,22.0,120.0,120.0,not anemic,fourth
699492,360472,38,3,2.196648,1383,1160,1383,pregnant,18.0,female,3.0,18.0,91.0,91.0,moderate,middle
699493,360472,39,4,2.196648,1383,1099,1383,pregnant,23.0,female,4.0,23.0,111.0,111.0,not anemic,middle


In [29]:
# https://stackoverflow.com/a/2415343/ with some tweaks
def weighted_avg_and_std(values, weights):
    """
    Return the weighted average and standard deviation.

    They weights are in effect first normalized so that they
    sum to 1 (and so they must not all be 0).

    values, weights -- NumPy ndarrays with the same shape.
    """
    is_nan = np.isnan(values)
    values = values[~is_nan]
    weights = weights[~is_nan]
    average = np.average(values, weights=weights)
    # Fast and numerically precise:
    variance = np.average((values - average) ** 2, weights=weights)
    return pd.Series(
        {
            "mean": average,
            "sd": np.sqrt(variance),
            # https://ngreifer.github.io/WeightIt/reference/ESS.html
            "effective_sample_size": (weights.sum() ** 2) / (weights**2).sum(),
        }
    )

In [30]:
# Matches table 10.21.1
pregnant_data[pregnant_data.anemia.notnull()].weight.sum()

np.float64(30325.760056)

In [31]:
# Within rounding error of table 10.21.1 value
weighted_avg_and_std(
    pregnant_data[pregnant_data.anemia.notnull()].anemia == "severe",
    pregnant_data[pregnant_data.anemia.notnull()].weight,
)

mean                         0.012959
sd                           0.113096
effective_sample_size    14669.827307
dtype: float64

In [32]:
# Within rounding error of table 10.21.1 value for any anemia
weighted_avg_and_std(
    pregnant_data[pregnant_data.anemia.notnull()].anemia.isin(
        ["severe", "moderate", "mild"]
    ),
    pregnant_data[pregnant_data.anemia.notnull()].weight,
)

mean                         0.503902
sd                           0.499985
effective_sample_size    14669.827307
dtype: float64

In [33]:
assert (
    (pregnant_data[pregnant_data.anemia.notnull()].anemia == "severe")
    == (pregnant_data[pregnant_data.anemia.notnull()].hemoglobin_adjusted < 70)
).all()

In [34]:
assert (
    (
        pregnant_data[pregnant_data.anemia.notnull()].anemia.isin(
            ["severe", "moderate", "mild"]
        )
    )
    == (pregnant_data[pregnant_data.anemia.notnull()].hemoglobin_adjusted < 110)
).all()

In [35]:
age_bin_edges = [15, 25, 30, 50]
age_bin_edges

[15, 25, 30, 50]

In [36]:
pregnant_data["age_group"] = pd.IntervalIndex(
    pd.cut(pregnant_data.age_hemoglobin, age_bin_edges, right=False)
)

In [37]:
# NOTE: We could use this; we just don't have the sample size for it in Nigeria
(
    pregnant_data.groupby(["age_group", "wealth_quintile"])
    .apply(lambda df: weighted_avg_and_std(df.hemoglobin_adjusted, weights=df.weight))
    .sort_index()
)

/tmp/ipykernel_2010209/3011596145.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pregnant_data.groupby(["age_group", "wealth_quintile"])
/tmp/ipykernel_2010209/3011596145.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: weighted_avg_and_std(df.hemoglobin_adjusted, weights=df.weight))


mean         sd  effective_sample_size
age_group wealth_quintile                                              
[15, 25)  lowest           106.243725  16.102944            2422.877457
          second           107.626974  15.769414            2474.651968
          middle           108.396657  15.178469            1534.594228
          fourth           109.889197  15.642170            1078.908505
          highest          112.223372  14.761249             974.612410
[25, 30)  lowest           106.276492  15.443291            1362.982979
          second           106.634183  16.739638            1055.621992
          middle           108.315369  15.777514             765.396213
          fourth           110.443527  15.356802             789.345556
          highest          113.866838  14.697004             681.730554
[30, 50)  lowest           104.988029  16.650049             915.790050
          second           105.732705  17.317991             539.946157
          middle           109.362365  17.694885             330.380565
          fourth           110.056847  15.658061             292.026753
          highest          113.840213  15.351691             405.429392

In [38]:
hemoglobin_disparities = (
    pregnant_data.groupby(["wealth_quintile"])
    .apply(lambda df: weighted_avg_and_std(df.hemoglobin_adjusted, weights=df.weight))
    .sort_index()
)
hemoglobin_disparities

/tmp/ipykernel_2010209/4277366928.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pregnant_data.groupby(["wealth_quintile"])
/tmp/ipykernel_2010209/4277366928.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: weighted_avg_and_std(df.hemoglobin_adjusted, weights=df.weight))


,mean,sd,effective_sample_size
wealth_quintile,,,
lowest,106.029126,16.025948,4689.833328
second,107.167072,16.209312,4047.784565
middle,108.455303,15.571656,2576.743299
fourth,110.079442,15.559339,2115.822472
highest,113.156279,14.858555,2014.207825


In [39]:
hemoglobin_disparities = hemoglobin_disparities.reset_index()
hemoglobin_disparities["sex"] = "Female"
hemoglobin_disparities = hemoglobin_disparities.set_index(["sex", "wealth_quintile"])
hemoglobin_disparities

mean         sd  effective_sample_size
sex    wealth_quintile                                              
Female lowest           106.029126  16.025948            4689.833328
       second           107.167072  16.209312            4047.784565
       middle           108.455303  15.571656            2576.743299
       fourth           110.079442  15.559339            2115.822472
       highest          113.156279  14.858555            2014.207825

In [40]:
results_dir = (
    "../results/pregnancies"
)

In [41]:
hemoglobin_disparities["mean"].rename("value").to_csv(
    f"{results_dir}/hemoglobin/mean_disparities/india.csv"
)

In [42]:
hemoglobin_disparities["sd"].rename("value").to_csv(
    f"{results_dir}/hemoglobin/sd_disparities/india.csv"
)

### Wealth quintile probabilities

Intuitively, you might think these would be equal; but we are looking at a subpopulation (pregnancies) that skews poorer.

In [43]:
assert pregnant_data.wealth_quintile.notnull().all()

In [44]:
wealth_quintile_probabilities = []

for quintile in pregnant_data.wealth_quintile.unique():
    quintile_info = pd.DataFrame(
        weighted_avg_and_std(
            pregnant_data.wealth_quintile == quintile, pregnant_data.weight
        )
    ).T
    quintile_info.insert(0, "wealth_quintile", quintile)
    wealth_quintile_probabilities.append(quintile_info)

wealth_quintile_probabilities = pd.concat(
    wealth_quintile_probabilities, ignore_index=True
)
wealth_quintile_probabilities.sort_values("mean")

,wealth_quintile,mean,sd,effective_sample_size
3,highest,0.165945,0.372032,14726.33105
2,fourth,0.181763,0.385649,14726.33105
0,middle,0.203904,0.402899,14726.33105
1,second,0.216223,0.411668,14726.33105
4,lowest,0.232164,0.422213,14726.33105


In [45]:
wealth_quintile_probabilities["mean"].sum()

np.float64(1.0)

In [46]:
wealth_quintile_probabilities = (
    wealth_quintile_probabilities.set_index("wealth_quintile")["mean"]
    .to_frame()
    .T.reset_index(drop=True)
)
wealth_quintile_probabilities.columns.name = None
wealth_quintile_probabilities

,middle,second,fourth,highest,lowest
0,0.203904,0.216223,0.181763,0.165945,0.232164


In [47]:
wealth_quintile_probabilities.insert(0, "sex", "Female")
wealth_quintile_probabilities

,sex,middle,second,fourth,highest,lowest
0,Female,0.203904,0.216223,0.181763,0.165945,0.232164


In [48]:
wealth_quintile_probabilities.to_csv(
    f"{results_dir}/wealth_quintile_probabilities/india.csv",
    index=False,
)

### Maternal mortality ratio

#### Maternal mortality rate

Not reported anywhere for India DHS, so we need to be extra careful since we can't cross-check.

In [49]:
adult_mortality_data.death_during_pregnancy_or_childbirth.value_counts()

death_during_pregnancy_or_childbirth
no     23283
yes      615
Name: count, dtype: int64

In [50]:
adult_mortality_data.death_violence_or_accident.value_counts()

death_violence_or_accident
no            67183
yes            7509
don't know      253
Name: count, dtype: int64

In [51]:
hhm_data

,cluster_number,household_number,weight,date_of_interview,line_number,currently_pregnant,age,sex,index_to_household,age_hemoglobin,wealth_quintile,hemoglobin_raw,hemoglobin_adjusted,anemia
0,10001,1,0.191072,1387,1,NaN,51.0,male,NaN,NaN,middle,NaN,NaN,NaN
1,10001,1,0.191072,1387,2,"not pregnant, don't know",46.0,female,2.0,46.0,middle,81.0,81.0,moderate
2,10001,1,0.191072,1387,3,NaN,22.0,male,NaN,NaN,middle,NaN,NaN,NaN
3,10001,1,0.191072,1387,4,"not pregnant, don't know",20.0,female,4.0,20.0,middle,113.0,113.0,mild
4,10001,9,0.191072,1387,1,"not pregnant, don't know",40.0,female,1.0,40.0,fourth,116.0,116.0,mild
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2869038,360482,85,2.270734,1385,1,NaN,60.0,male,NaN,NaN,middle,NaN,NaN,NaN
2869039,360482,85,2.270734,1385,2,NaN,50.0,female,NaN,NaN,middle,NaN,NaN,NaN
2869040,360482,96,2.270734,1385,1,NaN,66.0,male,NaN,NaN,middle,NaN,NaN,NaN
2869041,360482,96,2.270734,1385,2,"not pregnant, don't know",46.0,female,2.0,46.0,middle,119.0,119.0,mild


In [52]:
adult_mortality_data["age_at_death_years"] = (
    adult_mortality_data.age_at_death_unit.map(
        {"year": 1, "months": 1 / 12, "days": 1 / 365.25}
    )
    * adult_mortality_data.age_at_death
)

In [53]:
adult_mortality_data["adult_death"] = (
    adult_mortality_data.age_at_death_years >= 15
) & (adult_mortality_data.age_at_death_years < 50)

In [54]:
adult_mortality_data["date_of_death"] = adult_mortality_data.year_of_death.replace(
    {"don't know": np.nan}
).astype(float) + (
    adult_mortality_data.month_of_death.map(
        {
            "january": 1,
            "february": 2,
            "march": 3,
            "april": 4,
            "may": 5,
            "june": 6,
            "july": 7,
            "august": 8,
            "september": 9,
            "october": 10,
            "november": 11,
            "december": 12,
        }
    )
    - 0.5
) * (
    1 / 12
)

/tmp/ipykernel_2010209/2493017682.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  adult_mortality_data["date_of_death"] = adult_mortality_data.year_of_death.replace(


In [55]:
adult_mortality_data["date_of_birth"] = (
    adult_mortality_data.date_of_death - adult_mortality_data.age_at_death_years
)
adult_mortality_data["exposure_start"] = np.maximum(
    adult_mortality_data.date_of_birth + 15, 2011.0
)
adult_mortality_data["exposure_end"] = np.minimum(
    adult_mortality_data.date_of_birth + 50, adult_mortality_data.date_of_death
)
adult_mortality_data["exposure"] = np.maximum(
    adult_mortality_data.exposure_end - adult_mortality_data.exposure_start, 0
)
adult_mortality_data["weighted_exposure"] = (
    adult_mortality_data.weight * adult_mortality_data.exposure
)
adult_mortality_data

/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/0100_data_prep/.venv/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in maximum
  result = getattr(ufunc, method)(*inputs, **kwargs)
/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/0100_data_prep/.venv/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in minimum
  result = getattr(ufunc, method)(*inputs, **kwargs)
/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/0100_data_prep/.venv/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in maximum
  result = getattr(ufunc, method)(*inputs, **kwargs)


,weight,any_died,num_died,wealth_quintile,sex,age_at_death_unit,age_at_death,month_of_death,year_of_death,death_violence_or_accident,death_during_pregnancy_or_childbirth,age_at_death_years,adult_death,date_of_death,date_of_birth,exposure_start,exposure_end,exposure,weighted_exposure
0,0.008939,yes,1.0,middle,male,year,51.0,april,2015.0,no,NaN,51.0,False,2015.291667,1964.291667,2011.0,2014.291667,3.291667,0.029424
1,0.008939,yes,1.0,middle,male,months,56.0,april,2015.0,yes,NaN,4.666667,False,2015.291667,2010.625,2025.625,2015.291667,0,0.0
2,0.004495,yes,1.0,middle,male,year,45.0,august,2014.0,no,NaN,45.0,True,2014.625000,1969.625,2011.0,2014.625,3.625,0.016294
3,0.004495,yes,2.0,fourth,male,year,35.0,may,2014.0,no,NaN,35.0,True,2014.375000,1979.375,2011.0,2014.375,3.375,0.015171
4,0.004495,yes,2.0,fourth,male,year,40.0,august,2014.0,no,NaN,40.0,True,2014.625000,1974.625,2011.0,2014.625,3.625,0.016294
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74940,2.463153,yes,2.0,second,female,year,21.0,september,2013.0,yes,NaN,21.0,True,2013.708333,1992.708333,2011.0,2013.708333,2.708333,6.671039
74941,2.270734,yes,1.0,middle,female,year,80.0,november,2014.0,no,no,80.0,False,2014.875000,1934.875,2011.0,1984.875,0,0.0
74942,2.270734,yes,1.0,middle,male,year,50.0,january,2015.0,no,NaN,50.0,False,2015.041667,1965.041667,2011.0,2015.041667,4.041667,9.17755
74943,2.270734,yes,1.0,middle,male,months,45.0,january,2012.0,yes,NaN,3.75,False,2012.041667,2008.291667,2023.291667,2012.041667,0,0.0


In [56]:
living_exposure_data = hhm_data.copy()
living_exposure_data["date_of_birth"] = (
    (living_exposure_data.date_of_interview / 12) + 1900 - living_exposure_data.age
)
living_exposure_data["exposure_start"] = np.maximum(
    living_exposure_data.date_of_birth + 15, 2011.0
)
living_exposure_data["exposure_end"] = np.minimum(
    living_exposure_data.date_of_birth + 50,
    (living_exposure_data.date_of_interview / 12) + 1900,
)
living_exposure_data["exposure"] = np.maximum(
    living_exposure_data.exposure_end - living_exposure_data.exposure_start, 0
)
living_exposure_data["weighted_exposure"] = (
    living_exposure_data.weight * living_exposure_data.exposure
)
living_exposure_data

,cluster_number,household_number,weight,date_of_interview,line_number,currently_pregnant,age,sex,index_to_household,age_hemoglobin,wealth_quintile,hemoglobin_raw,hemoglobin_adjusted,anemia,date_of_birth,exposure_start,exposure_end,exposure,weighted_exposure
0,10001,1,0.191072,1387,1,NaN,51.0,male,NaN,NaN,middle,NaN,NaN,NaN,1964.583333,2011.0,2014.583333,3.583333,0.684675
1,10001,1,0.191072,1387,2,"not pregnant, don't know",46.0,female,2.0,46.0,middle,81.0,81.0,moderate,1969.583333,2011.0,2015.583333,4.583333,0.875747
2,10001,1,0.191072,1387,3,NaN,22.0,male,NaN,NaN,middle,NaN,NaN,NaN,1993.583333,2011.0,2015.583333,4.583333,0.875747
3,10001,1,0.191072,1387,4,"not pregnant, don't know",20.0,female,4.0,20.0,middle,113.0,113.0,mild,1995.583333,2011.0,2015.583333,4.583333,0.875747
4,10001,9,0.191072,1387,1,"not pregnant, don't know",40.0,female,1.0,40.0,fourth,116.0,116.0,mild,1975.583333,2011.0,2015.583333,4.583333,0.875747
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2869038,360482,85,2.270734,1385,1,NaN,60.0,male,NaN,NaN,middle,NaN,NaN,NaN,1955.416667,2011.0,2005.416667,0.000000,0.000000
2869039,360482,85,2.270734,1385,2,NaN,50.0,female,NaN,NaN,middle,NaN,NaN,NaN,1965.416667,2011.0,2015.416667,4.416667,10.029075
2869040,360482,96,2.270734,1385,1,NaN,66.0,male,NaN,NaN,middle,NaN,NaN,NaN,1949.416667,2011.0,1999.416667,0.000000,0.000000
2869041,360482,96,2.270734,1385,2,"not pregnant, don't know",46.0,female,2.0,46.0,middle,119.0,119.0,mild,1969.416667,2011.0,2015.416667,4.416667,10.029075


In [57]:
(adult_mortality_data.adult_death * adult_mortality_data.weight).sum()

np.float64(11897.673413)

In [58]:
adult_mortality_data.weighted_exposure.sum()

42011.871167166595

In [59]:
((adult_mortality_data.adult_death * adult_mortality_data.weight).sum()) / (
    adult_mortality_data.weighted_exposure.sum()
    + living_exposure_data.weighted_exposure.sum()
)

np.float64(0.0017707888221588613)

In [60]:
adult_mortality_data

,weight,any_died,num_died,wealth_quintile,sex,age_at_death_unit,age_at_death,month_of_death,year_of_death,death_violence_or_accident,death_during_pregnancy_or_childbirth,age_at_death_years,adult_death,date_of_death,date_of_birth,exposure_start,exposure_end,exposure,weighted_exposure
0,0.008939,yes,1.0,middle,male,year,51.0,april,2015.0,no,NaN,51.0,False,2015.291667,1964.291667,2011.0,2014.291667,3.291667,0.029424
1,0.008939,yes,1.0,middle,male,months,56.0,april,2015.0,yes,NaN,4.666667,False,2015.291667,2010.625,2025.625,2015.291667,0,0.0
2,0.004495,yes,1.0,middle,male,year,45.0,august,2014.0,no,NaN,45.0,True,2014.625000,1969.625,2011.0,2014.625,3.625,0.016294
3,0.004495,yes,2.0,fourth,male,year,35.0,may,2014.0,no,NaN,35.0,True,2014.375000,1979.375,2011.0,2014.375,3.375,0.015171
4,0.004495,yes,2.0,fourth,male,year,40.0,august,2014.0,no,NaN,40.0,True,2014.625000,1974.625,2011.0,2014.625,3.625,0.016294
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74940,2.463153,yes,2.0,second,female,year,21.0,september,2013.0,yes,NaN,21.0,True,2013.708333,1992.708333,2011.0,2013.708333,2.708333,6.671039
74941,2.270734,yes,1.0,middle,female,year,80.0,november,2014.0,no,no,80.0,False,2014.875000,1934.875,2011.0,1984.875,0,0.0
74942,2.270734,yes,1.0,middle,male,year,50.0,january,2015.0,no,NaN,50.0,False,2015.041667,1965.041667,2011.0,2015.041667,4.041667,9.17755
74943,2.270734,yes,1.0,middle,male,months,45.0,january,2012.0,yes,NaN,3.75,False,2012.041667,2008.291667,2023.291667,2012.041667,0,0.0


In [61]:
adult_mortality_data["maternal_death"] = (
    adult_mortality_data.adult_death
    & (adult_mortality_data.death_during_pregnancy_or_childbirth == "yes")
    & (adult_mortality_data.death_violence_or_accident != "yes")
)

In [62]:
(adult_mortality_data.maternal_death * adult_mortality_data.weight).sum()

np.float64(405.651753)

In [63]:
def maternal_mortality_rate(df):
    return ((df.maternal_death * df.weight).sum() * 1_000) / (
        df.weighted_exposure
    ).sum()

In [64]:
maternal_mortality_data = pd.concat(
    [
        adult_mortality_data[adult_mortality_data.sex == "female"][
            ["wealth_quintile", "maternal_death", "weight", "weighted_exposure"]
        ],
        living_exposure_data[living_exposure_data.sex == "female"][
            ["wealth_quintile", "weight", "weighted_exposure"]
        ].assign(maternal_death=False),
    ],
    ignore_index=True,
)

In [65]:
maternal_mortality_rate(maternal_mortality_data)

np.float64(0.1191121883940753)

In [66]:
maternal_mortality_rates = maternal_mortality_data.groupby("wealth_quintile").apply(
    maternal_mortality_rate
)
maternal_mortality_rates

/tmp/ipykernel_2010209/3795702594.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  maternal_mortality_rates = maternal_mortality_data.groupby("wealth_quintile").apply(
/tmp/ipykernel_2010209/3795702594.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  maternal_mortality_rates = maternal_mortality_data.groupby("wealth_quintile").apply(


wealth_quintile
lowest     0.251552
second     0.129580
middle     0.134920
fourth     0.076909
highest    0.034044
dtype: float64

#### General fertility rate

In [67]:
fertility_event_data = birth_data.copy()
fertility_event_data["birth_in_period"] = (
    (fertility_event_data.interview_date - fertility_event_data.birth_date) >= 1
) & ((fertility_event_data.interview_date - fertility_event_data.birth_date) <= 36)
fertility_event_data["weighted_birth_in_period"] = (
    fertility_event_data.birth_in_period * fertility_event_data.weight
)

In [68]:
fertility_event_data.weighted_birth_in_period.sum()

np.float64(150433.10539200003)

In [69]:
fertility_event_data.groupby("wealth_quintile").weighted_birth_in_period.sum()

/tmp/ipykernel_2010209/2169524943.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  fertility_event_data.groupby("wealth_quintile").weighted_birth_in_period.sum()


wealth_quintile
lowest     37315.912347
second     33131.370611
middle     30243.631415
fourth     27530.819389
highest    22211.371630
Name: weighted_birth_in_period, dtype: float64

In [70]:
fertility_exposure_data = wra_data.copy()
fertility_exposure_data["exposure_start"] = np.maximum(
    fertility_exposure_data.date_of_birth + 12 * 15,
    fertility_exposure_data.interview_date - 36,
)  # aka lowlim
# aka upplim
fertility_exposure_data["exposure_end"] = np.minimum(
    fertility_exposure_data.interview_date - 1,
    fertility_exposure_data.date_of_birth + 12 * 45,
)
fertility_exposure_data["exposure"] = (
    (fertility_exposure_data.exposure_end - fertility_exposure_data.exposure_start) + 1
).clip(lower=0)
fertility_exposure_data["weighted_exposure"] = (
    fertility_exposure_data.exposure * fertility_exposure_data.weight
)

In [71]:
gfr_by_wealth = (
    fertility_event_data.groupby("wealth_quintile").weighted_birth_in_period.sum()
    * 1_000
    / (fertility_exposure_data.groupby("wealth_quintile").weighted_exposure.sum() / 12)
)
gfr_by_wealth

/tmp/ipykernel_2010209/4103222212.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  fertility_event_data.groupby("wealth_quintile").weighted_birth_in_period.sum()
/tmp/ipykernel_2010209/4103222212.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  / (fertility_exposure_data.groupby("wealth_quintile").weighted_exposure.sum() / 12)


wealth_quintile
lowest     114.405014
second      91.556480
middle      79.049536
fourth      69.588896
highest     56.835044
dtype: float64

In [72]:
maternal_disorders_incidence_disparities = (
    (maternal_mortality_rates / gfr_by_wealth)
    .rename("value")
    .rename_axis("wealth_quintile")
    .reset_index()
)
maternal_disorders_incidence_disparities.insert(0, "sex", "Female")
maternal_disorders_incidence_disparities

,sex,wealth_quintile,value
0,Female,lowest,0.002199
1,Female,second,0.001415
2,Female,middle,0.001707
3,Female,fourth,0.001105
4,Female,highest,0.000599


In [73]:
maternal_disorders_incidence_disparities.to_csv(
    f"{results_dir}/maternal_disorders_incidence_disparities/india.csv",
    index=False,
)

## LBWSG

### Birth weight

In [74]:
birth_data["birth_weight_kilograms"] = birth_data.birth_weight_kilograms.replace(
    {"not weighed at birth": np.nan, "don't know": np.nan}
).astype(float)

/tmp/ipykernel_2010209/1262238226.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  birth_data["birth_weight_kilograms"] = birth_data.birth_weight_kilograms.replace(
/tmp/ipykernel_2010209/1262238226.py:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  birth_data["birth_weight_kilograms"] = birth_data.birth_weight_kilograms.replace(


In [75]:
weighted_avg_and_std(birth_data.birth_weight_kilograms, birth_data.weight)

mean                      2799.392719
sd                         604.338123
effective_sample_size    86484.148885
dtype: float64

In [76]:
birth_weight_disparities = birth_data.groupby("wealth_quintile").apply(
    lambda df: weighted_avg_and_std(df.birth_weight_kilograms, df.weight)
)
birth_weight_disparities

/tmp/ipykernel_2010209/890811655.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  birth_weight_disparities = birth_data.groupby("wealth_quintile").apply(


/tmp/ipykernel_2010209/890811655.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  birth_weight_disparities = birth_data.groupby("wealth_quintile").apply(


,mean,sd,effective_sample_size
wealth_quintile,,,
lowest,2767.050071,647.661212,23969.631404
second,2769.538275,608.623042,22126.156274
middle,2793.091464,594.535582,18888.148937
fourth,2811.885429,596.065203,14336.624949
highest,2861.478555,567.034576,12220.438182


In [77]:
birth_weight_disparities = (
    birth_weight_disparities["mean"].rename("value").reset_index()
)
birth_weight_disparities

,wealth_quintile,value
0,lowest,2767.050071
1,second,2769.538275
2,middle,2793.091464
3,fourth,2811.885429
4,highest,2861.478555


In [78]:
birth_weight_disparities.to_csv(
    f"{results_dir}/birth_weight_disparities/india.csv", index=False
)

### Short gestation

All we have here is a self-reported duration of pregnancy.

In [79]:
# Basically no difference in mean
birth_data.groupby("wealth_quintile").apply(
    lambda df: weighted_avg_and_std(df.duration_of_pregnancy, df.weight)
)

/tmp/ipykernel_2010209/2150176159.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  birth_data.groupby("wealth_quintile").apply(
/tmp/ipykernel_2010209/2150176159.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  birth_data.groupby("wealth_quintile").apply(


,mean,sd,effective_sample_size
wealth_quintile,,,
lowest,8.984584,0.506989,46757.969967
second,9.015559,0.518205,34263.878995
middle,9.050525,0.525370,25463.538749
fourth,9.051636,0.531054,18054.353166
highest,9.042706,0.535222,13759.568526


In [80]:
birth_data["short_gestation"] = np.where(
    birth_data.duration_of_pregnancy.isnull(),
    np.nan,
    birth_data.duration_of_pregnancy < 9.0,
)

In [81]:
weighted_avg_and_std(birth_data.short_gestation, birth_data.weight)

mean                          0.073412
sd                            0.260812
effective_sample_size    129817.319675
dtype: float64

In [82]:
birth_data.groupby("wealth_quintile").apply(
    lambda df: weighted_avg_and_std(df.short_gestation, df.weight)
)

/tmp/ipykernel_2010209/2497808234.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  birth_data.groupby("wealth_quintile").apply(
/tmp/ipykernel_2010209/2497808234.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  birth_data.groupby("wealth_quintile").apply(


,mean,sd,effective_sample_size
wealth_quintile,,,
lowest,0.078308,0.268655,46757.969967
second,0.070505,0.255996,34263.878995
middle,0.067307,0.250553,25463.538749
fourth,0.072144,0.258726,18054.353166
highest,0.079073,0.269852,13759.568526


The trends in short gestation don't make intuitive sense (?), so we do not plan to use them in the sim.